## SVOD_TEMPLATE_CREATION

###📍IMPORTANT: Always check the order of WBTV and Foundry data

In [5]:
# ============================================================
# 📚 LIBRARIES - EXTERNAL
# ============================================================
## Librarry:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import os
import time
from openpyxl import load_workbook
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
f = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh = logging.FileHandler('SVOD.log', mode='w')   # ✅ overwrite file
fh.setFormatter(f)
logger.addHandler(fh)

# ============================================================
# 📚 LIBRARIES - OWN FUNCTIONS
# ============================================================
from Packages import matching_pipeline
pipeline = matching_pipeline()

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_NO_TF"] = "1"

logger.info("Enter short and long synopsis word count thresholds")

while True:
    try:
        short = int(input("Enter SHORT synopsis limit: ").strip())
        long = int(input("Enter LONG synopsis limit: ").strip())

        if short <= 0 or long <= 0:
            logger.error(" Values must be positive integers. Try again.")
            continue

        break

    except ValueError:
        logger.error(" Invalid input. Please enter numeric values.")

template_type = pipeline.ask_content_type()

final_df = pipeline.run_template_pipeline(
    content_type=template_type,
    top_k=5,
    ce_threshold=0.75
)
Output_folder = pipeline.select_output_folder()
Format_check_path= os.path.join(Output_folder, "WBTVD or WB2B or FOUNDRY - Format.csv")
English_path = Format_check_path
Temp_path = Output_folder
final_df.to_csv(
    Format_check_path, index=False
)

logger.info("File verification Done")
verify = input("File verification Done [y/n]:")

if verify.lower() == "y":
    Only_English = input("The multilanguage Folder available? [y/n]: ")
    path = None
    if str(Only_English).lower() == "y":
        path = pipeline.pick_multilang_folder()
    Series_Name=input("Enter the Series name: ")
    Series = f"SVOD_{Series_Name}"
    en_path = English_path
    Temp = Temp_path

    output_file = os.path.join(Temp, f"{Series}_Template.xlsx")

    # =========================
    # DELETE OLD FILE
    # =========================
    if os.path.exists(output_file):
        try:
            os.remove(output_file)
        except PermissionError:
            raise RuntimeError("❌ Excel file is open. Close it and retry.")

    
    # =========================
    # LOAD ENGLISH MASTER
    # =========================
    English = pd.read_csv(en_path)

    if "Season" not in English.columns:
        English["Season"] = 0
    if "Episode" not in English.columns:
        English["Episode"] = 0

    English["Season"] = pd.to_numeric(English["Season"], errors="coerce").fillna(0).astype(int)
    English["Episode"] = pd.to_numeric(English["Episode"], errors="coerce").fillna(0).astype(int)

    if "Primary Release Date" in English.columns:
        English["Primary Release Date"] = pd.to_datetime(
            English["Primary Release Date"], errors="coerce"
        )

    if template_type == "series":
        English = English.sort_values(["Season", "Episode"]).reset_index(drop=True)
    else:
        title_col = pipeline.get_output_title_column(English)
        if title_col:
            English = English.sort_values(by=[title_col]).reset_index(drop=True)
        else:
            English = English.reset_index(drop=True)

    # =========================
    # EXCEL CREATION
    # =========================
    try:
        with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
            workbook = writer.book
            written_sheets = []

            # ====================================
            # ENGLISH ONLY
            # ====================================
            if str(Only_English).lower() == "n":
                df = English.copy()

                df["Sr.No."] = range(1, len(df) + 1)
                df["Category"] = "WB"
                title_col = pipeline.get_output_title_column(df)
                if title_col is None:
                    raise ValueError("❌ No valid title column found for output creation.")

                df["Source Title (Long Description)"] = df[title_col]

                if template_type == "series":
                    df["Source Title (Long Description)"] = [
                        f"{t}: Season {s}" if e == 0 and s != 0 else t
                        for t, e, s in zip(df["Source Title (Long Description)"], df["Episode"], df["Season"])
                    ]

                df["Localized Title"] = ""
                if "MPM Number" in df.columns:
                    df["WM Internal Reference"] = df["MPM Number"]
                elif "uuid" in df.columns:
                    df["WM Internal Reference"] = df["uuid"]
                else:
                    df["WM Internal Reference"] = ""
                if "Primary Release Date" in df.columns:
                    df["US Release Date"] = df["Primary Release Date"].dt.strftime("%Y-%m-%d")
                else:
                    df["US Release Date"] = ""

                df[f"Synopsis (Short) SOURCE ({short} Character Limit)"] = pipeline.char_limit(short, df, "yes")
                df[f"Synopsis (Short) TRANSLATION ({short} Characters Limit )"] = ""

                df[f"Synopsis (Long) SOURCE DATA ({long} Character Limit)"] = pipeline.char_limit(long, df, "yes")
                df[f"Synopsis (Long) TRANSLATION ({long} Character Limit)"] = ""

                row_count = len(df)
                df["Source char. Counter"] = [f"=LEN(I{i})" for i in range(2, row_count + 2)]
                df["Translation char. Count"] = [f"=LEN(K{i})" for i in range(2, row_count + 2)]
                df["Source Char. Counter"] = [f"=LEN(M{i})" for i in range(2, row_count + 2)]
                df["Translation Char. Count"] = [f"=LEN(O{i})" for i in range(2, row_count + 2)]

                final_columns = [
                    "Sr.No.", "Category", "Season", "Episode",
                    "Source Title (Long Description)", "Localized Title",
                    "WM Internal Reference", "US Release Date",
                    f"Synopsis (Short) SOURCE ({short} Character Limit)", "Source char. Counter",
                    f"Synopsis (Short) TRANSLATION ({short} Characters Limit )", "Translation char. Count",
                    f"Synopsis (Long) SOURCE DATA ({long} Character Limit)", "Source Char. Counter",
                    f"Synopsis (Long) TRANSLATION ({long} Character Limit)", "Translation Char. Count"
                ]
                for col in final_columns:
                    if col not in df.columns:
                        df[col] = ""
                df[final_columns].to_excel(writer, sheet_name="English", index=False)
                written_sheets.append("English")

            # ====================================
            # TRANSLATION FILES
            # ====================================
            else:
                final_columns = [
                        "Sr.No.", "Category", "Season", "Episode",
                        "Source Title (Long Description)", "Localized Title",
                        "WM Internal Reference", "US Release Date",
                        f"Synopsis (Short) SOURCE ({short} Character Limit)", "Source char. Counter",
                        f"Synopsis (Short) TRANSLATION ({short} Characters Limit )", "Translation char. Count",
                        f"Synopsis (Long) SOURCE DATA ({long} Character Limit)", "Source Char. Counter",
                        f"Synopsis (Long) TRANSLATION ({long} Character Limit)", "Translation Char. Count"
                    ]
                name_1="Info"
                for file in os.listdir(path):
                    if not file.endswith(".csv"):
                        continue
        
                    name = file.split("-")[2].split("export_")[1].replace("_", " ")
                    if name == "English United States":
                        continue
        
                    df = pd.read_csv(os.path.join(path, file))
        
                    df["Season"] = df["season-number"].fillna(0).astype(int)
                    df["Episode"] = df["episode-number"].fillna(0).astype(int)
                    df = df.sort_values(["Season", "Episode"]).reset_index(drop=True)
                    
                    df = pd.merge(
                        English[["Season", "Episode", "*Title name", "MPM Number", "Primary Release Date","uuid"]],
                        df,
                        on="uuid",
                        how="left"
                    )

                    if df["*Title name"].isna().any():
                        raise ValueError(f"❌ English mismatch in {name}")
        
                    df["Sr.No."] = range(1, len(df) + 1)
                    df["Category"] = "WB"
                    df["Season"] = df["Season_x"]
                    df["Episode"] = df["Episode_x"]
                    title_col = pipeline.get_output_title_column(df)
                    if title_col is None:
                        raise ValueError(f"❌ No title column found in translated sheet: {name}")

                    df["Source Title (Long Description)"] = df[title_col]
                    df["Localized Title"] = ""

                    if template_type == "series":
                        df["Source Title (Long Description)"] = pd.Series([
                            f"{i}: Season {int(s)}" if j == 0 and s != 0 else i
                            for i, j, s in zip(df["Source Title (Long Description)"], df["Episode"], df["Season"])
                        ])
                    df["WM Internal Reference"] = df["MPM Number"]
                    df["US Release Date"] = df["Primary Release Date"].dt.strftime("%Y-%m-%d")
        
                    df[f"Synopsis (Short) SOURCE ({short} Character Limit)"] = pipeline.char_limit(short, English, "yes")
                    df[f"Synopsis (Short) TRANSLATION ({short} Characters Limit )"] = pipeline.char_limit(short, df)
        
                    df[f"Synopsis (Long) SOURCE DATA ({long} Character Limit)"] = pipeline.char_limit(long, English, "yes")
                    df[f"Synopsis (Long) TRANSLATION ({long} Character Limit)"] = pipeline.char_limit(long, df)
        
                    row_count = len(df)
                    df["Source char. Counter"] = [f"=LEN(I{i})" for i in range(2, row_count + 2)]
                    df["Translation char. Count"] = [f"=LEN(K{i})" for i in range(2, row_count + 2)]
                    df["Source Char. Counter"] = [f"=LEN(M{i})" for i in range(2, row_count + 2)]
                    df["Translation Char. Count"] = [f"=LEN(O{i})" for i in range(2, row_count + 2)]
                
                
                    for col in final_columns:
                        if col not in df.columns:
                            df[col] = ""
                    df[final_columns].to_excel(writer, sheet_name=name, index=False)
                    written_sheets.append(name)
                                
                if not written_sheets:
                    pd.DataFrame({"Message": ["No valid translation data found"]}).to_excel(
                        writer, sheet_name="Info", index=False
                    )

    except PermissionError:
        raise RuntimeError("❌ Excel file is open. Close it before running.")


    workbook.save(output_file)
    time.sleep(5)
    # =========================
    # PATH CONFIGURATION
    # =========================
    summary_rows = []

    # =========================
    # MAIN PROCESSING
    # =========================

    sheets = pd.read_excel(output_file, sheet_name=None)

    for sheet_name, df in sheets.items():

        cols = df.columns.tolist()

        # Defaults
        col9_wc = 0
        col13_wc = 0

        col9_status = col13_status = "Not available"
        col11_status = col15_status = "Not available"
        col11_len_status = col14_len_status = "Not available"

        # =========================
        # COLUMN 11 (Translation Short Availability)
        # =========================
        if len(cols) >= 11:
            col11 = df.iloc[:, 10]
            col11_status = pipeline.availability_status(col11)
            col11_len_status = pipeline.synopsis_status(col11, short)

        # =========================
        # COLUMN 15 (Translation Long Availability)
        # =========================
        if len(cols) >= 15:
            col15 = df.iloc[:, 14]
            col15_status = pipeline.availability_status(col15)
            col14_len_status = pipeline.synopsis_status(col15, long)

        # =========================
        # COLUMN 9 (English Short)
        # =========================
        if len(cols) >= 9 and len(cols) >= 11:
            col9 = df.iloc[:, 8]
            col9_status = pipeline.synopsis_status(col9, short)
            col9_wc = pipeline.conditional_word_count(col9, col11)

        # =========================
        # COLUMN 13 (English Long)
        # =========================
        if len(cols) >= 13 and len(cols) >= 15:
            col13 = df.iloc[:, 12]
            col13_status = pipeline.synopsis_status(col13, long)
            col13_wc = pipeline.conditional_word_count(col13, col15)

        # =========================
        # APPEND SUMMARY
        # =========================
        summary_rows.append({
            "Sheet": sheet_name,
            f"Synopsis (Short) TRANSLATION ({short} Characters Limit)": col11_status,
            f"Synopsis (Long) TRANSLATION ({long} Character Limit)": col15_status,
            f"Synopsis (Short) ENGLISH ({short} Characters Limit)": col9_status,
            f"Synopsis (Long) ENGLISH ({long} Characters Limit)": col13_status,
            f"Synopsis (Short) TRANS_len ({short} Characters Limit)": col11_len_status,
            f"Synopsis (Long) TRANS_len ({long} Character Limit)": col14_len_status,
            f"Word_Count": col9_wc + col13_wc
        })

    # =========================
    # EXPORT
    # =========================
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df[summary_df["Sheet"] != "Info"]
    with pd.ExcelWriter(output_file, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        summary_df.to_excel(writer, sheet_name="Info", index=False)
    wb = load_workbook(output_file)

    for name in written_sheets:
        ws = wb[name]
        pipeline.format_sheet_openpyxl(ws)
    logger.info(f"Template '{Series}' created successfully.")


Select folder to save output files...
Selected Output Folder: C:/Users/mshanmugam/OneDrive - Warner Bros. Discovery/JUPYTER_PY/AI Projects/Automated Metadata Template Creation (SVOD)/Output_folder
